# Stage 1

In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import os
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.drill_mode import SoloDrillMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomPitchReset
from src.rl.reward_shapers import Stage1Reward
from src.rl.trainer import train_ppo_vectorized

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Training on: {device}")


def make_env():
    match_cfg = MatchConfig(
        mode=SoloDrillMode(),
        roster=[PlayerSlot(team="red", stats=PlayerStats(name="Agent", accel=3200.0))],
    )
    return HaxballGymEnv(
        match_config=match_cfg,
        reward_shaper=Stage1Reward(),
        reset_strategy=RandomPitchReset(min_distance=60.0),
        max_steps=900,  # 10s per episode at 60 FPS
    )


num_envs = 16
train_envs = gym.vector.AsyncVectorEnv([make_env for _ in range(num_envs)])
eval_env = make_env()

model = ActorCritic(obs_dim=80).to(device)

⚡ Training on: cuda


In [ ]:
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=10_000_000,
    num_envs=num_envs,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=100,
    save_dir="models/stage1",
    lr_initial=3.5e-4,
    lr_final=1e-5,
    ent_coef_initial=0.015,
    ent_coef_final=0.001,
)

train_envs.close()
eval_env.close()

🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:   3.0% (3/100) | Conceded:   2.0% (2/100) | Net:  +1 | Touch:  35.0% | Avg Steps: 882.1 | Mean Reward: -40.99
   ⭐ New verified best model saved: models/stage1/best_model.pt
      [Net: +1 | Scored: 3.0% | Reward: -40.99 | Speed: 882.1 steps]


📊 [EVALUATION @ Step  200704] Scored:   4.0% (4/100) | Conceded:   5.0% (5/100) | Net:  -1 | Touch:  64.0% | Avg Steps: 874.0 | Mean Reward: -48.54

📊 [EVALUATION @ Step  303104] Scored:  15.0% (15/100) | Conceded:   2.0% (2/100) | Net: +13 | Touch:  79.0% | Avg Steps: 844.4 | Mean Reward: -29.30
   ⭐ New verified best model saved: models/stage1/best_model.pt
      [Net: +13 | Scored: 15.0% | Reward: -29.30 | Speed: 844.4 steps]


📊 [EVALUATION @ Step  401408] Scored:   1.0% (1/100) | Conceded:   2.0% (2/100) | Net:  -1 | Touch:  32.0% | Avg Steps: 891.0 | Mean Reward: -45.51

📊 [EVALUATION @ Step  503808] Scored:   6.0% (6/100) | Conceded:   0.0% (0/10

# Testing

In [19]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load your RL Models
obs_dim = 80
stage1_model = ActorCritic(obs_dim).to(device)
stage1_model.load_state_dict(torch.load("models/stage1/best_model.pt", map_location=device))


# 2. Setup Team Coordinators
red_rl_controller = RLController(stage1_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(stage1_model, team="blue")

# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 60.0s per episode (5 Episodes)
   Episode 1: 10 goals
   Episode 2: 7 goals
   Episode 3: 11 goals
   Episode 4: 11 goals
   Episode 5: 9 goals
📊 Average Scoring Rate: 9.60 goals / 60.0s



9.6

In [21]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (1 Matches)
✅ Completed in 2.19s
🏆 Series Outcome (Wins): RED 1 | BLUE 0 | DRAWS 0
⚽ Avg Goals / Match:     RED 3.00 | BLUE 0.00



In [23]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: BLUE WINS! 🎉 (3 - 2)
Replay saved to: renders/arena/2026-08-22_04-45-13_match_1.html



In [24]:
from src.rl.benchmarker import render_solo_drill

device = torch.device("cpu")
model = ActorCritic(obs_dim=80).to(device)
model.load_state_dict(torch.load("models/stage1/best_model.pt", map_location=device, weights_only=False))

agent_slot = PlayerSlot("red", PlayerStats("RL_Agent"), RLController(model, team="red", device=device))

# Render 3 randomized episodes (20 seconds each)
replay_path = render_solo_drill(
    agent_slot=agent_slot,
    num_episodes=3,
    time_limit=20.0,
    save_path="renders/solo_drills",
)

🎬 Generating 3 Solo Drill Replays (20.0s each)...
   Episode 1 Finished: 3 Goals Scored
   Episode 2 Finished: 4 Goals Scored
   Episode 3 Finished: 2 Goals Scored
🏆 Overall: 3.00 Avg Goals / 20.0s
Replay saved to: renders/solo_drills/2026-08-22_04-47-03_solo_drill.html

